# Notebook 01 — Aquisição da malha viária, limite bairros e zona urbana (Overture Maps, IBGE)

**Projeto:** Acessibilidade Geográfica às UBS de Teresina: roteiro computacional AE2SFCA  
**Programa:** MAPEPROF - Mestrado Profissional em Planejamento Urbano e Regional / IFPI  
**Autor:** Felipe Ramos Dantas  
**Orientador:** Prof. Dr. Antonio Joaquim da Silva  
**Coorientador:** Prof. Dr. Reurysson Chagas de Sousa Morais  
**Repositório:** https://github.com/felipedantas-pi/mapeprof-accessibility-ubs  
**Última atualização:** 2026-05-06

## Objetivo

Baixar e pré-processar os dados geoespaciais e tabulares do **IBGE** necessários ao *pipeline* AE2SFCA. O processo engloba a delimitação da área de estudo (zona urbana + *buffer* de 5 km restrito ao limite municipal), a aquisição das geometrias censitárias e a extração e limpeza dos microdados do Censo 2022.

## Saída

| Arquivo | Fonte | Descrição |
|---|---|---|
| `teresina.parquet` | IBGE | Limite municipal de Teresina |
| `teresina_bairros.parquet` | IBGE | Malha de bairros oficiais |
| `teresina_zonaUrbana_utm.parquet` | IBGE | Perímetro da zona urbana (bairros dissolvidos) |
| `teresina_zonUrbanaClip_utm.parquet` | IBGE | Área de estudo: ZU + buffer 5 km ∩ limite municipal |
| `teresina_setoresCensitariosUrbanos.parquet` | IBGE | Setores censitários restritos à situação urbana |
| `teresina_gradeEstatistica_utm.parquet` | IBGE | Grade estatística intersecionada com o limite municipal |
| `teresina_setoresCensitarios_DadosCompletos.parquet`| IBGE | *GeoDataFrame* mestre fundido com os dados socioeconômicos |

## Pré-requisitos

- Conexão de internet estável (downloads da Overture podem chegar a centenas de MB).
- Notebook executado uma única vez por release da Overture.
- Tempo médio: ~10–15 min.

---

In [12]:
# ── 1. IMPORTAÇÕES E CONFIGURAÇÕES GLOBAIS ───────────────────────────────────

import pandas as pd
import geopandas as gpd
import requests
import zipfile
import io
from pathlib import Path

# Sistemas de Referência de Coordenadas (SRC)
SRC_METRICO    = "31983"   # SIRGAS 2000 / UTM Zona 23S (Cálculos de distância/área)
SRC_GEOGRAFICO = "4674"    # SIRGAS 2000 (Geográfico lat/lon)

# Estruturação automática de diretórios
DIR_BRUTOS = Path("../dados/brutos/ibge")
DIR_PROC   = Path("../dados/processados/ibge")
DIR_BRUTOS.mkdir(parents=True, exist_ok=True)
DIR_PROC.mkdir(parents=True, exist_ok=True)

print("✅ Dependências carregadas. SRC e diretórios configurados.")

✅ Dependências carregadas. SRC e diretórios configurados.


## 2. Aquisição das Malhas Territoriais e Definição da Área de Estudo

A área de estudo combina o perímetro da **zona urbana de Teresina** com um **buffer de 5 km**, recortada pelo **limite municipal**. Isso garante que a periferia próxima seja incluída na análise de acessibilidade sem extrapolar as fronteiras administrativas do município.

In [ ]:
# ── 2. DOWNLOAD E PRÉ-PROCESSAMENTO DAS MALHAS TERRITORIAIS ──────────────────

print("🌍 Baixando malhas territoriais do IBGE (Piauí)...")
url_mun = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2025/UFs/PI/PI_Municipios_2025.zip"
url_bairro = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/bairros/shp/UF/PI_bairros_CD2022.zip"

gdf_pi = gpd.read_file(f"zip+{url_mun}")
gdf_pi_bairro = gpd.read_file(f"zip+{url_bairro}")

print("✂️ Filtrando Teresina e estruturando a Zona Urbana...")
gdf_teresina = gdf_pi.query("NM_MUN == 'Teresina'").copy()
gdf_teresina_bairros = gdf_pi_bairro.query("NM_MUN == 'Teresina'").copy()

# Dissolve os limites internos dos bairros para gerar o polígono contínuo da Zona Urbana
gdf_zonaUrbana = gdf_teresina_bairros.dissolve()[['CD_MUN', 'NM_MUN', 'geometry']]

# Criação da Área de Estudo: Reprojeção Métrica -> Buffer 5km -> Reprojeção Geográfica -> Clip
gdf_zonaUrbana_5km = gdf_zonaUrbana.to_crs(SRC_METRICO)
gdf_zonaUrbana_5km['geometry'] = gdf_zonaUrbana_5km.buffer(distance=5000, cap_style='flat', join_style='bevel')
gdf_zonaUrbana_5km = gdf_zonaUrbana_5km.to_crs(SRC_GEOGRAFICO)

# Recorta (clip) a zona urbana expandida mantendo-a estritamente dentro de Teresina
gdf_zonaUrbana_5km_clip = gpd.clip(gdf_zonaUrbana_5km, gdf_teresina)

# Exportação em formato colunar de alta performance (GeoParquet)
print("💾 Exportando arquivos territoriais...")
gdf_teresina.to_crs(SRC_METRICO).to_parquet(DIR_BRUTOS / "teresina.parquet", index=False)
gdf_teresina_bairros.to_crs(SRC_METRICO).to_parquet(DIR_BRUTOS / "teresina_bairros.parquet", index=False)
gdf_zonaUrbana.to_crs(SRC_METRICO).to_parquet(DIR_BRUTOS / "teresina_zonaUrbana_utm.parquet", index=False)
gdf_zonaUrbana_5km_clip.to_crs(SRC_METRICO).to_parquet(DIR_BRUTOS / "teresina_zonaUrbanaClip_utm.parquet", index=False)

print("✅ Malhas territoriais processadas com sucesso.")

🌍 Baixando malhas territoriais do IBGE (Piauí)...
✂️ Filtrando Teresina e estruturando a Zona Urbana...
💾 Exportando arquivos territoriais...
✅ Malhas territoriais processadas com sucesso.


## 3. Aquisição dos Recortes Espaciais Censitários

Obtenção das geometrias de alta resolução do IBGE para alocação espacial da demanda. A **Grade Estatística** passará por um `overlay` espacial para reter estritamente os dados internos ao município de Teresina, eliminando o peso topológico do arquivo nacional.

In [3]:
# ── 3. DOWNLOAD E CRUZAMENTO ESPACIAL (SETORES E GRADE) ──────────────────────

print("📊 Baixando Setores Censitários e Grade Estatística do IBGE...")
url_setores = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/setores/shp/UF/PI_setores_CD2022.zip"
url_grade = "https://geoftp.ibge.gov.br/recortes_para_fins_estatisticos/grade_estatistica/censo_2022/grade_estatistica/grade_id66.zip"

# Isola apenas os setores classificados como "Urbana" em Teresina
gdf_teresina_scUrbano = gpd.read_file(f"zip+{url_setores}").query("NM_MUN == 'Teresina' and SITUACAO == 'Urbana'")

# Lê a grade estatística completa
gdf_gradeEstatistica = gpd.read_file(f"zip+{url_grade}")

print("✂️ Cruzando Grade Estatística com os limites de Teresina (Overlay)...")
# O overlay funde as geometrias e herda apenas as colunas essenciais de Teresina
gdf_grade_intersec = gpd.overlay(
    gdf_gradeEstatistica, 
    gdf_teresina[['CD_MUN', 'NM_MUN', 'geometry']], 
    how='intersection'
)

print("💾 Exportando arquivos censitários...")
gdf_teresina_scUrbano.to_crs(SRC_METRICO).to_parquet(DIR_BRUTOS / "teresina_setoresCensitariosUrbanos.parquet", index=False)
gdf_grade_intersec.to_crs(SRC_METRICO).to_parquet(DIR_BRUTOS / "teresina_gradeEstatistica_utm.parquet", index=False)

print("✅ Geometrias censitárias processadas com sucesso.")

📊 Baixando Setores Censitários e Grade Estatística do IBGE...
✂️ Cruzando Grade Estatística com os limites de Teresina (Overlay)...
💾 Exportando arquivos censitários...
✅ Geometrias censitárias processadas com sucesso.


## 4. Pipeline ETL: Dados Tabulares e Construção do GeoDataFrame Mestre

Para popular a geometria dos setores urbanos com atributos socioeconômicos (demanda do modelo AE2SFCA), extraímos tabelas compactadas diretamente na memória RAM. Uma função automatizada padroniza as nomenclaturas, lida com problemas tipográficos do IBGE (`CD_setor` vs `CD_SETOR`) e substitui os caracteres de sigilo estatístico por zero numérico.

In [ ]:
# ── 4. DECLARAÇÃO DE FUNÇÕES E URLS DO PIPELINE ETL ──────────────────────────

URL_BASE_SETORES = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Setor_csv/"
URL_BASE_RENDA   = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios_Rendimento_do_Responsavel/"

# Dicionário de configuração das variáveis extraídas
TABELAS_CENSO = {
    'corRaca': {
        'url': f"{URL_BASE_SETORES}Agregados_por_setores_cor_ou_raca_BR.zip",
        'colunas': ['CD_SETOR', 'V01317', 'V01318', 'V01319', 'V01320', 'V01321']
    },
    'demog': {
        'url': f"{URL_BASE_SETORES}Agregados_por_setores_demografia_BR.zip",
        'colunas': ['CD_SETOR', 'V01006', 'V01031', 'V01032', 'V01033', 'V01034', 'V01035', 'V01036', 'V01037', 'V01038', 'V01039', 'V01040', 'V01041']
    },
    'renda': {
        'url': f"{URL_BASE_RENDA}Agregados_por_setores_renda_responsavel_BR_20260508_csv.zip",
        'colunas': ['CD_SETOR', 'V06003', 'V06004', 'V06005', 'V06006']
    }
}

def extrair_e_limpar_censo(url, colunas_alvo, sufixo):
    """
    Realiza o download em memória (io.BytesIO), filtra colunas, resolve
    inconsistências de uppercase/lowercase e limpa os dados censurados do IBGE.
    """
    print(f"  -> Processando grupo: {sufixo.upper()}...")
    resposta = requests.get(url)

    # Trava o código de forma limpa e mostra a URL exata se o IBGE não retornar sucesso (Código 200)
    if resposta.status_code != 200:
        raise ValueError(f"\n❌ Erro de Conexão: O arquivo não foi encontrado no servidor.\nO IBGE retornou o código HTTP {resposta.status_code}.\nVerifique no navegador se o nome ou a pasta do arquivo mudou.\nLink tentado: {url}")
    
    with zipfile.ZipFile(io.BytesIO(resposta.content), 'r') as z:
        nome_csv = [n for n in z.namelist() if n.endswith('.csv')][0]
        with z.open(nome_csv) as f_csv:
            # Leitura de cabeçalho para mapeamento de case sensitivity
            amostra = pd.read_csv(f_csv, sep=';', nrows=0)
            colunas_arquivo = amostra.columns.tolist()
            mapa_colunas = {col.upper(): col for col in colunas_arquivo}
            
            # Ajusta as colunas alvo para o formato exato que existe no arquivo
            colunas_reais_para_ler = [mapa_colunas[c] for c in colunas_alvo]
            
            # Leitura da tabela completa com filtro de uso e tipagem estrita
            f_csv.seek(0)
            df = pd.read_csv(
                f_csv, 
                sep=';', 
                usecols=colunas_reais_para_ler,
                dtype={mapa_colunas['CD_SETOR']: str},
                low_memory=False
            )
            
    # Padroniza todas as colunas para caixa alta (resolve o 'CD_setor' do IBGE)
    df.rename(str.upper, axis='columns', inplace=True)
    
    # Renomeia adicionando o sufixo (blindando a chave primária CD_SETOR)
    dict_renomear = {c: f"{c}_{sufixo}" for c in colunas_alvo if c != 'CD_SETOR'}
    df.rename(columns=dict_renomear, inplace=True)
    
    # Limpeza estatística: Converte 'X' e '-' para 0 numérico (int/float)
    cols_limpar = list(dict_renomear.values())
    for col in cols_limpar:
        df[col] = pd.to_numeric(df[col].replace(['X', '-'], 0), errors='coerce').fillna(0)
        
    return df

In [11]:
# ── 5. EXECUÇÃO DO PIPELINE E MERGE ESPACIAL ─────────────────────────────────

print("📥 Iniciando extração remota de dados tabulares (Censo 2022)...")

# O GeoDataFrame base que receberá as junções sucessivas
gdf_master = gdf_teresina_scUrbano.copy()

for chave, config in TABELAS_CENSO.items():
    # Roda a extração, filtragem e limpeza na memória RAM
    df_temp = extrair_e_limpar_censo(config['url'], config['colunas'], chave)
    
    # O 'how=left' garante que a malha de Teresina permaneça inalterada
    # Descartando automaticamente todas as outras linhas do Brasil no df_temp
    gdf_master = gdf_master.merge(df_temp, on='CD_SETOR', how='left')

# Exportação final reprojetada para análise métrica AE2SFCA
caminho_final = DIR_PROC / "teresina_setoresCensitarios_DadosCompletos.parquet"
gdf_master.to_crs(SRC_METRICO).to_parquet(caminho_final, index=False)

print("\n🎉 Pipeline concluído com sucesso!")
print(f"O GeoDataFrame mestre possui {len(gdf_master)} setores censitários e {len(gdf_master.columns)} atributos.")
print(f"Salvo em: {caminho_final}")

📥 Iniciando extração remota de dados tabulares (Censo 2022)...
  -> Processando grupo: CORRACA...
  -> Processando grupo: DEMOG...
  -> Processando grupo: RENDA...

🎉 Pipeline concluído com sucesso!
O GeoDataFrame mestre possui 1403 setores censitários e 51 atributos.
Salvo em: ..\dados\processados\ibge\teresina_setoresCensitarios_DadosCompletos.parquet
